In [0]:
%run ../config/config

In [0]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F

spark.conf.set("spark.databricks.execution.timeout", "18000")

SILVER_TABLE_PATH = f"{CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}"
GOLD_TABLE_PREFIX = f"{CATALOG}.{GOLD_SCHEMA}"

In [0]:
df_main = spark.table(SILVER_TABLE_PATH)

df_main = df_main.filter(
    F.col("is_current") == True
)

df_main = (
    df_main.withColumn(
        "entry_id",
        F.xxhash64("product_id")
    ).withColumn(
        "brand_id",
        F.xxhash64("brands_list")
    )
)

fact_product_entries = (
    df_main
    .select(*FACT_PRODUCT_ENTRIES_COLS)
    .distinct()
)

dim_brands = (
    df_main
    .select(*DIM_BRANDS_COLS)
    .distinct()
)

dim_products = (
    df_main
    .select(*DIM_PRODUCTS_COLS)
    .distinct()
)

window = Window.partitionBy("brand_id", "brands_list") \
      .orderBy(
          F.when((F.col("brand_owner").isNull()) |
        (F.col("brand_owner") == "N/A"), 1).otherwise(0)
      )

dim_brands = dim_brands.withColumn(
  'rn',
  F.row_number() \
    .over(window)
).filter(F.col('rn') == 1).drop('rn')

In [0]:
df_dict = {
    GOLD_FACT_PRODUCT_ENTRIES: fact_product_entries,
    GOLD_DIM_BRANDS: dim_brands,
    GOLD_DIM_PRODUCTS: dim_products
}

for name, df in df_dict.items():
    (
        df.write
        .mode('overwrite')
        .format('delta')
        .option('mergeSchema', 'true')
        .saveAsTable(f"{GOLD_TABLE_PREFIX}.{name}")
    )